# Custom High-Scale Wake Word Training Pipeline

**Target phrase**: Hello DJ  
**Model name**: Hello_DJ (auto-derived from target_phrase spaces).  
**Scale**: Matched to hey_jarvis production config - 200k positive samples, 50k training steps.

In [ ]:
# Step 1: Download MIT RIRs and setup background audio datasets
import os, shutil
from huggingface_hub import hf_hub_download, list_repo_files
from tqdm import tqdm

repo_id = "davidscripka/MIT_environmental_impulse_responses"
output_dir = "/home/jovyan/mit_rirs/16khz"
os.makedirs(output_dir, exist_ok=True)

print("Downloading MIT environmental impulse responses...")
rir_files = [f for f in list_repo_files(repo_id, repo_type='dataset')
             if f.startswith('16khz/') and f.endswith('.wav')]
for fname in tqdm(rir_files, desc="MIT RIRs"):
    hf_hub_download(repo_id=repo_id, filename=fname, repo_type='dataset', local_dir="/home/jovyan/mit_rirs")

n_rirs = len([f for f in os.listdir(output_dir) if f.endswith('.wav')])
print(f"mit_rirs: {n_rirs} wav files in 16khz/")

os.makedirs('/tmp/audioset_16k', exist_ok=True)
src_pos = '/home/jovyan/open-wakeword-repo/notebooks/training_tutorial_data/positive'
if os.path.isdir(src_pos):
    for w in [f for f in os.listdir(src_pos) if f.endswith('.wav')]:
        shutil.copy(os.path.join(src_pos, w), '/tmp/audioset_16k/')
print(f'audioset_16k: {len(os.listdir("/tmp/audioset_16k"))} wav files')

impulse_src = '/home/jovyan/piper-sample-generator/impulses'
if os.path.isdir(impulse_src):
    for w in os.listdir(impulse_src):
        if w.endswith('.wav'):
            shutil.copy(os.path.join(impulse_src, w), output_dir)
print(f'mit_rirs after impulses copy: {len([f for f in os.listdir(output_dir) if f.endswith(".wav")])} wav files')

In [ ]:
# Step 2: Download pre-computed openWakeWord features
import os
from huggingface_hub import hf_hub_download

HF_TOKEN = None
print("Downloading pre-computed openWakeWord features...")

hf_hub_download(repo_id="davidscripka/openwakeword_features", filename="validation_set_features.npy",
                repo_type="dataset", token=HF_TOKEN, local_dir="/home/jovyan")
hf_hub_download(repo_id="davidscripka/openwakeword_features", filename="openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
                repo_type="dataset", token=HF_TOKEN, local_dir="/home/jovyan")

print("Pre-computed features downloaded:")
val_path = "/home/jovyan/validation_set_features.npy"
feat_path = "/home/jovyan/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
print(f"  validation_set_features.npy: {os.path.getsize(val_path) / 1e6:.1f} MB")
print(f"  openwakeword_features_ACAV100M_2000_hrs_16bit.npy: {os.path.getsize(feat_path) / 1e6:.1f} MB")

## Step 3: Setup openwakeword and dependencies

In [ ]:
# Step 3a: Install openwakeword, patch piper, and download model config
import sys, os, subprocess

# Clone openwakeword source if not present
oww_dir = "/home/jovyan/openwakeword"
if not os.path.isdir(oww_dir):
    print("Cloning openWakeWord...")
    subprocess.run(["git", "clone", "--depth=1",
        "https://github.com/dscripka/openWakeWord.git", oww_dir], check=True)
else:
    print("openWakeWord source already present")

sys.path.insert(0, oww_dir)

try:
    import openwakeword
    print(f"openwakeword already installed at {openwakeword.__file__}")
except (ImportError, AttributeError):
    get_ipython().system("pip install -e /home/jovyan/openwakeword 2>&1 | tail -5")

get_ipython().system("pip install --user --break-system-packages webrtcvad espeak-phonemizer 2>&1 | tail -3")
get_ipython().system("pip install 'numpy<2' piper-tts 2>&1 | tail -3")
get_ipython().system("pip install --user --break-system-packages onnxscript onnx_tf tensorflow 2>&1 | tail -3")
get_ipython().system("pip install --user --break-system-packages 'scipy<1.17' 2>&1 | tail -3")
get_ipython().system("pip install --user --break-system-packages onnxruntime-gpu 2>&1 | tail -3")
print("Dependencies OK")

# Clone piper-sample-generator if not present
psg_dir = "/home/jovyan/piper-sample-generator"
if not os.path.isdir(psg_dir):
    print("Cloning piper-sample-generator...")
    subprocess.run(["git", "clone", "--depth=1",
        "https://github.com/dscripka/piper-sample-generator.git", psg_dir], check=True)
else:
    print("piper-sample-generator already present")

# Patch generate_samples.py for weights_only=False
gen_path = os.path.join(psg_dir, "generate_samples.py")
with open(gen_path, 'r') as f:
    content = f.read()
content = content.replace('torch.load(model_path)', 'torch.load(model_path, weights_only=False)')
with open(gen_path, 'w') as f:
    f.write(content)
print("Patched generate_samples.py for weights_only=False")

# Download model JSON config if missing
import requests
models_dir = os.path.join(psg_dir, "models")
json_path = os.path.join(models_dir, "en-us-libritts-high.pt.json")
if not os.path.exists(json_path) or os.path.getsize(json_path) == 0:
    print("Downloading model JSON config...")
    url = 'https://raw.githubusercontent.com/rhasspy/piper-sample-generator/v2.0.0/models/en-us-libritts-high.pt.json'
    r = requests.get(url)
    os.makedirs(models_dir, exist_ok=True)
    with open(json_path, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded {len(r.content)} bytes")
else:
    print(f"Model JSON config exists ({os.path.getsize(json_path)} bytes)")

# Download LibriTTS model .pt file if missing
model_pt = os.path.join(models_dir, "en-us-libritts-high.pt")
if not os.path.exists(model_pt):
    print("Downloading LibriTTS model (~1.3 GB)...")
    url = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'
    get_ipython().system("wget -O '" + model_pt + "' '" + url + "' 2>&1 | tail -5")
    print(f"Downloaded model ({os.path.getsize(model_pt) / 1e9:.1f} GB)")
else:
    print(f"Model already exists ({os.path.getsize(model_pt) / 1e9:.1f} GB)")

# Fix data.py to use exist_ok
import_path = os.path.join(oww_dir, "openwakeword", "data.py")
with open(import_path, 'r') as f:
    content = f.read()
content = content.replace(
    'os.mkdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), "resources"))',
    'os.makedirs(os.path.join(os.path.dirname(os.path.abspath(__file__)), "resources"), exist_ok=True)')
with open(import_path, 'w') as f:
    f.write(content)
print("Patched data.py with exist_ok=True")

# Patch data.py for stereo RIRs + resampling
with open(import_path, 'r') as f:
    content = f.read()
old = 'rir_waveform, sr = torchaudio.load(random.choice(RIR_paths))\n            augmented_batch = reverberate(augmented_batch.cpu(), rir_waveform, rescale_amp="avg")'
new = 'rir_waveform, sr = torchaudio.load(random.choice(RIR_paths))\n            # Convert stereo/multi-channel to mono\n            if rir_waveform.shape[0] > 1:\n                rir_waveform = rir_waveform.mean(dim=0, keepdim=True)\n            # Resample to 16kHz if needed\n            if sr != 16000:\n                rir_waveform = torchaudio.functional.resample(rir_waveform, sr, 16000)\n                sr = 16000\n            augmented_batch = reverberate(augmented_batch.cpu(), rir_waveform, rescale_amp="avg")'
if old in content:
    content = content.replace(old, new)
    with open(import_path, 'w') as f:
        f.write(content)
    print("Patched data.py for stereo RIRs + resampling")
else:
    print("data.py RIR patch already applied")

# Patch train.py opset_version and use dynamo=False for proper ONNX export
train_path = os.path.join(oww_dir, "openwakeword", "train.py")
with open(train_path, 'r') as f:
    content = f.read()
content = content.replace('opset_version=13', 'opset_version=17')
content = content.replace('opset_version=17)', 'opset_version=17, dynamo=False)')
with open(train_path, 'w') as f:
    f.write(content)
print("Patched train.py opset_version and dynamo=False for proper ONNX export")

# Copy generate_samples.py and piper_train into openwakeword
get_ipython().system("cp /home/jovyan/piper-sample-generator/generate_samples.py /home/jovyan/openwakeword/openwakeword/generate_samples.py")
get_ipython().system("cp -r /home/jovyan/piper-sample-generator/piper_train /home/jovyan/openwakeword/openwakeword/piper_train")
print("Copied generate_samples.py and piper_train module")

# Download feature models (melspectrogram + embedding) for openWakeWord
res_dir = os.path.join(oww_dir, "openwakeword", "resources", "models")
os.makedirs(res_dir, exist_ok=True)
feature_urls = [
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx",
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite",
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx",
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite",
]
for url in feature_urls:
    fname = url.split("/")[-1]
    path = os.path.join(res_dir, fname)
    if not os.path.exists(path):
        print(f"Downloading {fname}...")
        get_ipython().system("wget -O '" + path + "' '" + url + "' 2>&1 | tail -3")
        print(f"  {os.path.getsize(path) / 1e6:.1f} MB")
    else:
        print(f"{fname} already exists ({os.path.getsize(path) / 1e6:.1f} MB)")
print("Feature models ready")

# Ensure my_custom_model.yml exists with all required keys
import yaml

yml_path = "/home/jovyan/my_custom_model.yml"
required_defaults = {
    'model_name': 'Hello_DJ',
    'target_phrase': ['Hello DJ'],
    'n_samples': 200000,
    'n_samples_val': 5000,
    'steps': 50000,
    'model_type': 'dnn',
    'layer_size': 64,
    'augmentation_rounds': 2,
    'tts_batch_size': 8,
    'augmentation_batch_size': 16,
    'piper_sample_generator_path': '/home/jovyan/piper-sample-generator',
    'mit_rir_dir': '/home/jovyan/mit_rirs/16khz',
    'rir_paths': ['/home/jovyan/mit_rirs/16khz'],
    'background_paths': ['/tmp/audioset_16k'],
    'background_paths_duplication_rate': [1],
    'false_positive_validation_data_path': '/home/jovyan/validation_set_features.npy',
    'output_dir': '/home/jovyan/Hello_DJ',
    'target_false_positives_per_hour': 0.2,
    'max_negative_weight': 1500,
    'feature_data_files': {
        'ACAV100M_sample': '/home/jovyan/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
    },
    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50,
    },
    'positive_train_output_path': '',
    'positive_test_output_path': '',
    'negative_train_output_path': '',
    'negative_test_output_path': '',
    'custom_negative_phrases': [],
}

if os.path.exists(yml_path):
    with open(yml_path) as f:
        config = yaml.safe_load(f.read())
    for k, v in required_defaults.items():
        if k not in config:
            config[k] = v
    config['tts_batch_size'] = 8
    with open(yml_path, 'w') as f:
        yaml.dump(config, f)
    print(f"Config file updated with any missing keys at {yml_path}")
else:
    with open(yml_path, 'w') as f:
        yaml.dump(required_defaults, f)
    print(f"Generated default config at {yml_path}")

In [ ]:
# Step 3b: Read, update, and save training config
import yaml, os

yml_path = '/home/jovyan/my_custom_model.yml'
if not os.path.exists(yml_path):
    print(f"Config file not found at {yml_path}, generating defaults")
    config = {
        'model_name': 'Hello_DJ',
        'target_phrase': ['Hello DJ'],
        'n_samples': 200000,
        'steps': 50000,
        'model_type': 'dnn',
        'layer_size': 64,
        'augmentation_rounds': 2,
        'tts_batch_size': 32,
        'piper_sample_generator_path': '/home/jovyan/piper-sample-generator',
        'mit_rir_dir': '/home/jovyan/mit_rirs/16khz',
        'rir_paths': ['/home/jovyan/mit_rirs/16khz'],
        'background_paths': ['/tmp/audioset_16k'],
        'background_paths_duplication_rate': [1],
        'false_positive_validation_data_path': '/home/jovyan/validation_set_features.npy',
        'output_dir': '/home/jovyan/Hello_DJ',
        'target_false_positives_per_hour': 0.2,
        'max_negative_weight': 1500,
    }
else:
    with open(yml_path) as f:
        config = yaml.safe_load(f.read())

print("Current config:")
print(yaml.dump(config))

config.update({
    'piper_sample_generator_path': '/home/jovyan/piper-sample-generator',
    'mit_rir_dir': '/home/jovyan/mit_rirs/16khz',
    'rir_paths': ['/home/jovyan/mit_rirs/16khz'],
    'background_paths': ['/tmp/audioset_16k'],
    'background_paths_duplication_rate': [1],
    'false_positive_validation_data_path': '/home/jovyan/validation_set_features.npy',
    'output_dir': '/home/jovyan/Hello_DJ',
    'model_name': 'Hello_DJ',
    'target_phrase': ['Hello DJ'],
    'n_samples': 200000,
    'steps': 50000,
    'model_type': 'dnn',
    'layer_size': 64,
    'augmentation_rounds': 2,
    'tts_batch_size': 32,
    'target_false_positives_per_hour': 0.2,
    'max_negative_weight': 1500,
})

with open(yml_path, 'w') as f:
    yaml.dump(config, f)

print("\nUpdated config saved:")
print(yaml.dump(config))

## Step 4: Generate synthetic training clips

Runs Piper TTS to generate 200k positive samples of "Hello DJ" and adversarial negative samples.

In [ ]:
# Step 4a: Generate positive and negative synthetic clips
import sys

sys.path.insert(0, '/home/jovyan/openwakeword')
sys.path.insert(0, '/home/jovyan/piper-sample-generator')

!LD_LIBRARY_PATH=/usr/local/cuda-12.9/compat:$LD_LIBRARY_PATH     PYTHONPATH=/home/jovyan/openwakeword:/home/jovyan/piper-sample-generator:$PYTHONPATH     python3 /home/jovyan/hellodj/augment_wrapper.py     --training_config /home/jovyan/my_custom_model.yml     --generate_clips

In [ ]:
# Step 4b: Augment clips with RIRs and background noise
import sys

sys.path.insert(0, '/home/jovyan/openwakeword')
sys.path.insert(0, '/home/jovyan/piper-sample-generator')

!LD_LIBRARY_PATH=/usr/local/cuda-12.9/compat:$LD_LIBRARY_PATH     PYTHONPATH=/home/jovyan/openwakeword:/home/jovyan/piper-sample-generator:$PYTHONPATH     python3 /home/jovyan/hellodj/augment_wrapper.py     --training_config /home/jovyan/my_custom_model.yml     --augment_clips

Computing features:  32%|████▋          | 3944/12500 [26:51<1:05:31,  2.18it/s]

Computing features:  32%|████▋          | 3945/12500 [26:52<1:07:44,  2.11it/s]

Computing features:  32%|████▋          | 3946/12500 [26:53<1:32:56,  1.53it/s]

Computing features:  32%|████▋          | 3947/12500 [26:53<1:26:26,  1.65it/s]

Computing features:  32%|████▋          | 3948/12500 [26:54<1:21:07,  1.76it/s]

...

Computing features:  32%|█████▍           | 3968/12500 [27:02<58:28,  2.43it/s]

Computing features:  32%|█████▍           | 3969/12500 [27:03<57:27,  2.47it/s]

Computing features:  32%|█████▍           | 3970/12500 [27:03<56:41,  2.51it/s]

Computing features:  32%|█████▍           | 3971/12500 [27:04<56:36,  2.51it/s]

Computing features:  32%|█████▍           | 3972/12500 [27:04<59:56,  2.37it/s]

## Step 5: Train the model

Requires augmented features from Step 4b.

In [ ]:
# Step 5a: Train the model (50k steps, 64-unit layers)
import sys

sys.path.insert(0, '/home/jovyan/openwakeword')
sys.path.insert(0, '/home/jovyan/piper-sample-generator')

!LD_LIBRARY_PATH=/usr/local/cuda/compat:$LD_LIBRARY_PATH     PYTHONPATH=/home/jovyan/openwakeword:/home/jovyan/piper-sample-generator:$PYTHONPATH     python3 /home/jovyan/hellodj/augment_wrapper.py     --training_config /home/jovyan/my_custom_model.yml     --train_model

In [ ]:
# Step 5b: Verify outputs
import os, glob

print("=== Output check ===")
model_dir = '/home/jovyan/Hello_DJ/Hello_DJ'
if os.path.isdir(model_dir):
    for sub in ['positive_train', 'positive_test', 'negative_train', 'negative_test']:
        path = os.path.join(model_dir, sub)
        n = len(glob.glob(os.path.join(path, '*.wav'))) if os.path.isdir(path) else 0
        print(f"  {sub}: {n} wav files")

onnx_path = os.path.join('/home/jovyan/Hello_DJ', 'Hello_DJ.onnx')
tflite_path = os.path.join('/home/jovyan/Hello_DJ', 'Hello_DJ.tflite')
if os.path.exists(onnx_path):
    sz = os.path.getsize(onnx_path)
    print(f"  ONNX model: {onnx_path} ({sz/1024:.1f}KB)")
if os.path.exists(tflite_path):
    print(f"  TFLite model: {tflite_path} ({os.path.getsize(tflite_path) / 1e6:.1f} MB)")
print("Done")